before running any HPC Workload.  a supercomputing engeeniner must inspect the underlying physical silicone in this step we probe the host prosessor core count , hardware threds cashe line sizes and vector units

In [ ]:

import os
import subprocess

def run_command(cmd):
    result = subprocess.run(
        cmd,
        shell=True,
        capture_output=True,
        text=True
    )
    return result.stdout.strip()


cpu_model = run_command("lscpu | grep 'Model name' | cut -d: -f2")
cores = run_command("lscpu | grep 'Core(s) per socket' | cut -d: -f2")
sockets = run_command("lscpu | grep 'Socket(s)' | cut -d: -f2")
threads = run_command("lscpu | grep 'Thread(s) per core' | cut -d: -f2")
numa_nodes = run_command("lscpu | grep 'NUMA node(s)' | cut -d: -f2")
cache_size = run_command("lscpu | grep 'L3 cache' | cut -d: -f2")
vector_units = run_command("lscpu | grep '^CPU(s):' | cut -d: -f2")

print(f"CPU Model: {cpu_model}")
print(f"Cores per Socket: {cores}")
print(f"Sockets: {sockets}")
print(f"Threads per Core: {threads}")
print(f"NUMA Nodes: {numa_nodes}")
print(f"L3 Cache Size: {cache_size}")
print(f"Logical CPUs: {vector_units}")



CPU Model: Intel(R) Xeon(R) CPU @ 2.00GHz
Cores per Socket: 1
Sockets: 1
Threads per Core: 2
NUMA Nodes: 1
L3 Cache Size: 38.5 MiB (1 instance)
Logical CPUs: 2


In [ ]:
from genericpath import exists
import os
#1 Create a simple execute shell script

with open('student_tutorial.sh','w') as f:
    f.write(''' #!/bin/bash
    echo "=== System Info ==="
    echo "User: $(whoami)"
    echo "Path: $(pwd)"
    ''')
os.chmod('student_tutorial.sh', 0o755)
!./student_tutorial.sh

#2. Generate SSH keys Simply

if not os.path.exists('/root/.ssh/id_rsa'):
    !mkdir -p ~/.ssh && ssh-keygen -t rsa -b 4096 -f ~/.ssh/id_rsa -N ""

#3. Create simple github Actions template

os.makedirs("./github/workflows",exist_ok=True)
with open('./github/workflows/ci.yml','w') as f:
    f.write(''' name: Test Script
  on: [push]
  jobs:
    run-script:
      runs on: ubuntu-latest
      steps:
      - uses: actions/checkout@v4
      - run: chmod +x ./student_tutorial.sh && ./student_tutorial.sh
    ''')
print("\n SetUp compelete! Simple files are ready")
print("=========================================")
print("Copy your PUBLIC SSH KEY BELOW")
print("=========================================")
with open('/root/.ssh/id_rsa.pub') as f:
    print(f.read().strip())
print("===========================================")

=== System Info ===
User: root
Path: /content
Generating public/private rsa key pair.
Your identification has been saved in /root/.ssh/id_rsa
Your public key has been saved in /root/.ssh/id_rsa.pub
The key fingerprint is:
SHA256:Bg4mczxajRCmcet1UjcXnVuSb5w0EeRALZ8CJImre5U root@df04ddf4ab2a
The key's randomart image is:
+---[RSA 4096]----+
|. =.  ..+o++o=+o |
| = + +...+. *o=  |
|. + @ +.   . X.+ |
| . O *..    o B  |
|  o  .. S.   o   |
|    .  .E        |
|     . .         |
|    . .          |
|     .           |
+----[SHA256]-----+

 SetUp compelete! Simple files are ready
Copy your PUBLIC SSH KEY BELOW
ssh-rsa AAAAB3NzaC1yc2EAAAADAQABAAACAQDNdKlApYgfHPBounznhcFpVokUERhAXQxr6PC61OtvIsg/jlTvkYT6HGsCMud1svLZqN16Gx0+yMBdLQYAFSAqteSWExWWaGYtxmq1v5evJAEAcEVdYzBEsIyb1Qjx4I4aqmX91hQzfACw2Vr2OWcsUne3oVcUdogyPbx+z8FJfh8atx5Yo+JOBQOJKK1o9vTDpDQF/mECGAs0iwtXyJFJmI8OY82WWwmxxga9OEdSauw7nxv+ehVfE8Mad7e76fu/U1WlFsFsEYWwUpUosd+26f/to4PxJtLVvss7sRfR8wtyT/wH7xN0NbVt2GcGUK9x03VIZM+vnYRfOICJPCeRpFQaXY

In [ ]:
import os

# 1. Fill your GitHub details below:
GITHUB_USERNAME = "bhavyatan"
GITHUB_EMAIL = "bhavya.t145@gmail.com"
REPO_NAME = "hpc-training-ssh-setup"

print("⚙️ Configuring Git details...")

os.system(f'git config --global user.name "{GITHUB_USERNAME}"')
os.system(f'git config --global user.email "{GITHUB_EMAIL}"')

print("✓ Git credentials configured locally!")

print("\n📁 Initializing Git Repository locally...")
os.system('git init')
os.system('git branch -M main')

print("✓ Git Repository initialized.")

# Adding code to remote
os.system(
    f'git remote add origin '
    f'https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git'
)

# Add files to staging
os.system('git add .')

# Commit changes
os.system('git commit -m "Initial commit"')

# Push code to GitHub
os.system('git push -u origin main')

⚙️ Configuring Git details...
✓ Git credentials configured locally!

📁 Initializing Git Repository locally...
✓ Git Repository initialized.


32768

In [ ]:
!ssh -o StrictHostKeyChecking=no -T git@github.com

git@github.com: Permission denied (publickey).


In [ ]:
import os

print("[*] Checking for existing Intel oneAPI installation...")
has_oneapi = os.path.exists("/opt/intel/oneapi")

if not has_oneapi:
    print("[*] Configuring Intel Official Package Repository...")
    commands = '''
    set -e
    wget -qO - https://apt.repos.intel.com/intel-gpg-keys/GPG-PUB-KEY-INTEL-SW-PRODUCTS.PUB | gpg --dearmor | sudo tee /usr/share/keyrings/oneapi-archive-keyring.gpg > /dev/null
    echo "deb [signed-by=/usr/share/keyrings/oneapi-archive-keyring.gpg] https://apt.repos.intel.com/oneapi all main" | sudo tee /etc/apt/sources.list.d/oneAPI.list
    sudo apt-get update -q
    # Install standalone oneAPI DPC++/C++ compiler and MPI developer headers
    sudo apt-get install -y -q intel-oneapi-compiler-dpcpp-cpp intel-oneapi-mpi-devel build-essential
    '''
    # We execute setup in background / check readiness
    print("[+] Intel repository commands prepared.")
else:
    print("[+] Intel oneAPI is already installed at /opt/intel/oneapi!")

[*] Checking for existing Intel oneAPI installation...
[*] Configuring Intel Official Package Repository...
[+] Intel repository commands prepared.


In [ ]:
import subprocess

# Execute the prepared installation commands in the shell environment
if not has_oneapi:
    print("[*] Installing Intel oneAPI packages (this might take a few minutes)... ")
    process = subprocess.Popen(commands, shell=True, executable='/bin/bash', stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

    # Stream output in real-time
    for line in iter(process.stdout.readline, ''):
        print(line, end='')

    process.stdout.close()
    return_code = process.wait()
    if return_code == 0:
        print("\n[+] Intel oneAPI environment installed successfully! Please re-run the compilation cells.")
    else:
        print(f"\n[!] Installation failed with exit code: {return_code}")
else:
    print("[+] Intel oneAPI is already available.")

[*] Installing Intel oneAPI packages (this might take a few minutes)... 
deb [signed-by=/usr/share/keyrings/oneapi-archive-keyring.gpg] https://apt.repos.intel.com/oneapi all main
Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  InRelease [1,578 B]
Get:3 https://apt.repos.intel.com/oneapi all InRelease [5,684 B]
Get:4 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:5 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Get:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  Packages [2,002 kB]
Get:7 https://apt.repos.intel.com/oneapi all/main amd64 Packages [960 kB]
Hit:8 http://archive.ubuntu.com/ubuntu noble InRelease
Get:9 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Get:11 https://apt.repos.intel.com/oneapi all/main all 

In [ ]:
import subprocess

# Execute the prepared installation commands in the shell environment
if not has_oneapi:
    print("[*] Installing Intel oneAPI packages (this might take a few minutes)... ")
    process = subprocess.Popen(commands, shell=True, executable='/bin/bash', stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

    # Stream output in real-time
    for line in iter(process.stdout.readline, ''):
        print(line, end='')

    process.stdout.close()
    return_code = process.wait()
    if return_code == 0:
        print("\n[+] Intel oneAPI environment installed successfully! Please re-run the compilation cells.")
    else:
        print(f"\n[!] Installation failed with exit code: {return_code}")
else:
    print("[+] Intel oneAPI is already available.")

[*] Installing Intel oneAPI packages (this might take a few minutes)... 
deb [signed-by=/usr/share/keyrings/oneapi-archive-keyring.gpg] https://apt.repos.intel.com/oneapi all main
Hit:1 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  InRelease
Hit:4 https://apt.repos.intel.com/oneapi all InRelease
Hit:5 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:6 https://r2u.stat.illinois.edu/ubuntu noble InRelease
Hit:7 http://archive.ubuntu.com/ubuntu noble InRelease
Hit:8 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:9 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Hit:11 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu noble InRelease
Reading package lists...
W: Skipping acquire of configured file 'main/sour

In [ ]:
import os
os.makedirs('/opt/intel/oneapi', exist_ok=True)

# Create a mock/redirecting setvars.sh that exports paths to icpx and system-installed oneAPI tools
setvars_content = """#!/bin/bash
# Automatically exported configurations for oneAPI components
export PATH=/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin
export PATH=$PATH:$(dirname $(which icpx 2>/dev/null || echo '/usr/bin'))
echo ':: oneAPI environment initialized manually ::'
"""

with open('/opt/intel/oneapi/setvars.sh', 'w') as f:
    f.write(setvars_content)

print('[+] Successfully generated /opt/intel/oneapi/setvars.sh')

[+] Successfully generated /opt/intel/oneapi/setvars.sh


In [ ]:
!bash -c "source /opt/intel/oneapi/setvars.sh && icpx -fsycl sycl_device_query.cpp -o sycl_device_query && ./sycl_device_query"

:: oneAPI environment initialized manually ::
bash: line 1: icpx: command not found


In [ ]:
import os
import subprocess

# Let's locate the actual vars.sh or icpx path under /opt/intel/oneapi
print("Searching for icpx or setvars in /opt/intel...")
paths = []
for root, dirs, files in os.walk("/opt/intel"):
    for file in files:
        if file in ["setvars.sh", "vars.sh", "icpx"]:
            paths.append(os.path.join(root, file))

print("Found matching files:")
for p in paths:
    print(p)

Searching for icpx or setvars in /opt/intel...
Found matching files:
/opt/intel/oneapi/setvars.sh
/opt/intel/oneapi/mpi/2021.19/env/vars.sh
/opt/intel/oneapi/mpi/2021.19/etc/mpi/vars.sh
/opt/intel/oneapi/libfabric/2.6/env/vars.sh
/opt/intel/oneapi/libfabric/2.6/etc/libfabric/vars.sh
/opt/intel/oneapi/dev-utilities/2026.2/env/vars.sh
/opt/intel/oneapi/umf/1.2/env/vars.sh
/opt/intel/oneapi/tcm/1.6/env/vars.sh
/opt/intel/oneapi/compiler/2026.2/bin/icpx
/opt/intel/oneapi/compiler/2026.2/env/vars.sh
/opt/intel/oneapi/compiler/2026.2/etc/compiler/vars.sh
/opt/intel/oneapi/debugger/2026.2/env/vars.sh
/opt/intel/oneapi/debugger/2026.2/etc/debugger/vars.sh
/opt/intel/oneapi/dpl/2022.14/env/vars.sh
/opt/intel/oneapi/dpl/2022.14/etc/dpl/vars.sh
/opt/intel/oneapi/tbb/2023.2/env/vars.sh
/opt/intel/oneapi/tbb/2023.2/etc/tbb/vars.sh


In [ ]:
# Create a simple sycl_device_query.cpp to compile and test
cpp_code = """#include <sycl/sycl.hpp>
#include <iostream>

int main() {
    for (auto device : sycl::device::get_devices()) {
        std::cout << "Device: " << device.get_info<sycl::info::device::name>() << std::endl;
        std::cout << "  Type: " << (device.is_gpu() ? "GPU" : (device.is_cpu() ? "CPU" : "Host/Other")) << std::endl;
    }
    return 0;
}
"""

with open('sycl_device_query.cpp', 'w') as f:
    f.write(cpp_code)

print("[+] Created sycl_device_query.cpp")

[+] Created sycl_device_query.cpp


In [ ]:
# Compile and run using the real compiler variables environment
import subprocess

# We discovered the compiler vars are at /opt/intel/oneapi/compiler/2026.2/env/vars.sh
cmd = """bash -c 'source /opt/intel/oneapi/compiler/2026.2/env/vars.sh && icpx -fsycl sycl_device_query.cpp -o sycl_device_query && ./sycl_device_query'"""

result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
print("STDOUT:")
print(result.stdout)
print("STDERR:")
print(result.stderr)

STDOUT:
Device:            Intel(R) Xeon(R) CPU @ 2.00GHz
  Type: CPU

STDERR:



In [ ]:
import subprocess

# Source the correct environment and execute the compiled query binary
cmd = """bash -c 'source /opt/intel/oneapi/compiler/2026.2/env/vars.sh && icpx -fsycl sycl_device_query.cpp -o sycl_device_query && ./sycl_device_query'"""
result = subprocess.run(cmd, shell=True, capture_output=True, text=True)

print("--- SYCL Device Query Output ---")
if result.returncode == 0:
    print(result.stdout)
else:
    print("Error during execution:")
    print(result.stderr)

--- SYCL Device Query Output ---
Device:            Intel(R) Xeon(R) CPU @ 2.00GHz
  Type: CPU



In [ ]:
import os

# Let's locate the 'sycl/sycl.hpp' header file under /opt/intel/
print("Searching for sycl/sycl.hpp...")
sycl_headers = []
for root, dirs, files in os.walk("/opt/intel"):
    if "sycl.hpp" in files:
        sycl_headers.append(os.path.join(root, "sycl.hpp"))

if sycl_headers:
    print("[+] Found SYCL headers at:")
    for header in sycl_headers:
        print(header)
else:
    print("[-] SYCL header 'sycl.hpp' not found. Please ensure compiler is fully installed.")

Searching for sycl/sycl.hpp...
[+] Found SYCL headers at:
/opt/intel/oneapi/compiler/2026.2/include/sycl/sycl.hpp
/opt/intel/oneapi/compiler/2026.2/include/CL/sycl.hpp


In [ ]:
%%writefile sycl_device_query.cpp
// ==============================================================================
// Program: sycl_device_query.cpp
// Purpose: Introspect SYCL platforms, compute devices, and memory capabilities
// ==============================================================================
#include <iostream>
#include <vector>

#if __has_include(<sycl/sycl.hpp>)
#include <sycl/sycl.hpp>
#define HAS_SYCL 1
#elif __has_include(<CL/sycl.hpp>)
#include <CL/sycl.hpp>
#define HAS_SYCL 1
#else
#define HAS_SYCL 0
#endif

int main() {
    std::cout << "==========================================================" << std::endl;
    std::cout << "      INTEL oneAPI / SYCL COMPUTE DEVICE DISCOVERY        " << std::endl;
    std::cout << "==========================================================" << std::endl;

#if HAS_SYCL
    // Enumerate all platforms
    auto platforms = sycl::platform::get_platforms();
    std::cout << "[+] Found " << platforms.size() << " SYCL Platform(s):" << std::endl;

    for (size_t p = 0; p < platforms.size(); ++p) {
        std::cout << "\n  Platform [" << p << "]: "
                  << platforms[p].get_info<sycl::info::platform::name>()
                  << " (" << platforms[p].get_info<sycl::info::platform::vendor>() << ")"
                  << std::endl;

        auto devices = platforms[p].get_devices();
        for (size_t d = 0; d < devices.size(); ++d) {
            std::cout << "    └── Device [" << d << "]: "
                      << devices[d].get_info<sycl::info::device::name>() << std::endl;
            std::cout << "        • Compute Units     : "
                      << devices[d].get_info<sycl::info::device::max_compute_units>() << std::endl;
            std::cout << "        • Max Work-Group Size: "
                      << devices[d].get_info<sycl::info::device::max_work_group_size>() << std::endl;
            std::cout << "        • Global Memory     : "
                      << (devices[d].get_info<sycl::info::device::global_mem_size>() / (1024 * 1024))
                      << " MB" << std::endl;
        }
    }
#else
    std::cout << "[-] SYCL headers not found. Running Native C++20 Multicore Host Inspector:" << std::endl;
    std::cout << "    • Host Architecture: x86_64 Multicore Processor" << std::endl;
    std::cout << "    • Execution Target : Host CPU SIMD Vector Engine" << std::endl;
#endif

    std::cout << "==========================================================" << std::endl;
    return 0;
}

Overwriting sycl_device_query.cpp


In [ ]:
import os
import glob

# Find the actual path where icpx resides inside /opt/intel/oneapi
icpx_paths = glob.glob('/opt/intel/oneapi/compiler/*/bin/icpx')
if not icpx_paths:
    icpx_paths = glob.glob('/opt/intel/oneapi/compiler/*/linux/bin/intel64/icpx')

if icpx_paths:
    icpx_path = icpx_paths[0]
    compiler_bin_dir = os.path.dirname(icpx_path)
    print(f"[+] Found icpx compiler at: {icpx_path}")

    # Generate a robust setvars.sh referencing the actual installation paths
    setvars_content = f"""#!/bin/bash
export PATH={compiler_bin_dir}:$PATH
export LD_LIBRARY_PATH=/opt/intel/oneapi/compiler/latest/lib:$LD_LIBRARY_PATH
echo ':: oneAPI Compiler Paths Added ::'
"""
    with open('/opt/intel/oneapi/setvars.sh', 'w') as f:
        f.write(setvars_content)
    print("[+] /opt/intel/oneapi/setvars.sh updated successfully!")
else:
    print("[-] Could not find icpx automatically. Let's list the /opt/intel/oneapi directories to check where it is installed:")
    !find /opt/intel/oneapi -name "icpx"

[+] Found icpx compiler at: /opt/intel/oneapi/compiler/latest/bin/icpx
[+] /opt/intel/oneapi/setvars.sh updated successfully!


In [ ]:
%%writefile sycl_device_query.cpp
// ==============================================================================
// Program: sycl_device_query.cpp
// Purpose: Introspect SYCL platforms, filtering and counting only those containing "Intel"
// ==============================================================================
#include <iostream>
#include <vector>
#include <string>

#if __has_include(<sycl/sycl.hpp>)
#include <sycl/sycl.hpp>
#define HAS_SYCL 1
#elif __has_include(<CL/sycl.hpp>)
#include <CL/sycl.hpp>
#define HAS_SYCL 1
#else
#define HAS_SYCL 0
#endif

int main() {
    std::cout << "==========================================================" << std::endl;
    std::cout << "      INTEL-FILTERED SYCL COMPUTE DEVICE DISCOVERY        " << std::endl;
    std::cout << "==========================================================" << std::endl;

#if HAS_SYCL
    auto platforms = sycl::platform::get_platforms();
    size_t intel_platform_count = 0;

    for (const auto& platform : platforms) {
        std::string platform_name = platform.get_info<sycl::info::platform::name>();
        if (platform_name.find("Intel") != std::string::npos) {
            intel_platform_count++;
        }
    }

    std::cout << "[+] Found " << intel_platform_count << " Intel SYCL Platform(s):" << std::endl;

    size_t p_idx = 0;
    for (const auto& platform : platforms) {
        std::string platform_name = platform.get_info<sycl::info::platform::name>();
        if (platform_name.find("Intel") != std::string::npos) {
            std::cout << "\n  Platform [" << p_idx++ << "]: "
                      << platform_name
                      << " (" << platform.get_info<sycl::info::platform::vendor>() << ")"
                      << std::endl;

            auto devices = platform.get_devices();
            for (size_t d = 0; d < devices.size(); ++d) {
                std::cout << "    └── Device [" << d << "]: "
                          << devices[d].get_info<sycl::info::device::name>() << std::endl;
                std::cout << "        • Compute Units     : "
                          << devices[d].get_info<sycl::info::device::max_compute_units>() << std::endl;
                std::cout << "        • Max Work-Group Size: "
                          << devices[d].get_info<sycl::info::device::max_work_group_size>() << std::endl;
                std::cout << "        • Global Memory     : "
                          << (devices[d].get_info<sycl::info::device::global_mem_size>() / (1024 * 1024))
                          << " MB" << std::endl;
            }
        }
    }
#else
    std::cout << "[-] SYCL headers not found." << std::endl;
#endif

    std::cout << "==========================================================" << std::endl;
    return 0;
}

Overwriting sycl_device_query.cpp


In [ ]:
import os
import subprocess

# Ensure oneAPI compiler is present
if not os.path.exists("/opt/intel/oneapi/compiler"):
    print("[*] Re-configuring and installing Intel oneAPI standalone compiler...")
    setup_cmds = """
    set -e
    wget -qO - https://apt.repos.intel.com/intel-gpg-keys/GPG-PUB-KEY-INTEL-SW-PRODUCTS.PUB | gpg --dearmor | sudo tee /usr/share/keyrings/oneapi-archive-keyring.gpg > /dev/null
    echo "deb [signed-by=/usr/share/keyrings/oneapi-archive-keyring.gpg] https://apt.repos.intel.com/oneapi all main" | sudo tee /etc/apt/sources.list.d/oneAPI.list
    sudo apt-get update -q
    sudo apt-get install -y -q intel-oneapi-compiler-dpcpp-cpp intel-oneapi-mpi-devel build-essential
    """
    subprocess.run(setup_cmds, shell=True, executable='/bin/bash')

# Dynamically locate the compiler's vars.sh dynamically
import glob
vars_paths = glob.glob('/opt/intel/oneapi/compiler/*/env/vars.sh')
if vars_paths:
    vars_sh = vars_paths[0]
    print(f"[+] Found vars.sh at: {vars_sh}")
    cmd = f"""bash -c 'source {vars_sh} && icpx -fsycl sycl_device_query.cpp -o sycl_device_query && ./sycl_device_query'"""
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print("STDOUT:")
    print(result.stdout)
    print("STDERR:")
    print(result.stderr)
else:
    print("[-] Failed to locate Intel compiler environment files after installation check.")

[+] Found vars.sh at: /opt/intel/oneapi/compiler/latest/env/vars.sh
STDOUT:
      INTEL-FILTERED SYCL COMPUTE DEVICE DISCOVERY        
[+] Found 1 Intel SYCL Platform(s):

  Platform [0]: Intel(R) OpenCL (Intel(R) Corporation)
    └── Device [0]:            Intel(R) Xeon(R) CPU @ 2.00GHz
        • Compute Units     : 2
        • Max Work-Group Size: 8192
        • Global Memory     : 12975 MB

STDERR:



In [ ]:
%%writefile sycl_vector_mul.cpp
// ==============================================================================
// Program: sycl_vector_mul.cpp
// Purpose: Perform element-wise multiplication on 5M elements using SYCL USM
// ==============================================================================
#include <sycl/sycl.hpp>
#include <iostream>
#include <vector>
#include <cmath>

constexpr size_t N = 5000000; // 5 Million elements

int main() {
    // Create a default selector to target available accelerator or CPU
    sycl::queue q(sycl::default_selector_v);
    std::cout << "[+] Running on device: "
              << q.get_device().get_info<sycl::info::device::name>() << std::endl;

    // Allocate Unified Shared Memory (USM) accessible by both host and device
    float* A = sycl::malloc_shared<float>(N, q);
    float* B = sycl::malloc_shared<float>(N, q);
    float* C = sycl::malloc_shared<float>(N, q);

    if (A == nullptr || B == nullptr || C == nullptr) {
        std::cerr << "[-] USM Allocation Failed!" << std::endl;
        return -1;
    }

    // Initialize vectors on the Host
    for (size_t i = 0; i < N; ++i) {
        A[i] = static_cast<float>(i);
        B[i] = 2.0f;
        C[i] = 0.0f;
    }

    std::cout << "[+] Executing element-wise vector multiplication on " << N << " elements..." << std::endl;

    // Submit the parallel kernel to the queue
    auto kernel_event = q.parallel_for(sycl::range<1>(N), [=](sycl::id<1> idx) {
        size_t i = idx[0];
        C[i] = A[i] * B[i];
    });

    // Wait for the kernel execution to complete
    kernel_event.wait();

    // Verify the results on the Host
    bool verification_passed = true;
    for (size_t i = 0; i < N; ++i) {
        float expected = static_cast<float>(i) * 2.0f;
        if (std::abs(C[i] - expected) > 1e-5f) {
            std::cout << "[-] Verification failed at index " << i
                      << ": Expected " << expected << ", got " << C[i] << std::endl;
            verification_passed = false;
            break;
        }
    }

    if (verification_passed) {
        std::cout << "[+] Verification PASSED! C[i] = A[i] * B[i] correctly calculated for 5,000,000 elements." << std::endl;
        // Print a subset of results
        std::cout << "    Sample Results:" << std::endl;
        for (size_t i = 0; i < 5; ++i) {
            std::cout << "    └── Index " << i << ": " << A[i] << " * " << B[i] << " = " << C[i] << std::endl;
        }
        std::cout << "    └── ..." << std::endl;
        std::cout << "    └── Index " << N - 1 << ": " << A[N - 1] << " * " << B[N - 1] << " = " << C[N - 1] << std::endl;
    }

    // Free Allocated USM Memory
    sycl::free(A, q);
    sycl::free(B, q);
    sycl::free(C, q);

    return 0;
}

Writing sycl_vector_mul.cpp


In [ ]:
import subprocess

# Compile and run the Vector Multiplication code utilizing our known vars.sh path
cmd = """bash -c 'source /opt/intel/oneapi/compiler/2026.2/env/vars.sh && icpx -fsycl sycl_vector_mul.cpp -o sycl_vector_mul && ./sycl_vector_mul'"""
result = subprocess.run(cmd, shell=True, capture_output=True, text=True)

print("STDOUT:")
print(result.stdout)
print("STDERR:")
print(result.stderr)

STDOUT:
[+] Running on device:            Intel(R) Xeon(R) CPU @ 2.00GHz
[+] Executing element-wise vector multiplication on 5000000 elements...
[+] Verification PASSED! C[i] = A[i] * B[i] correctly calculated for 5,000,000 elements.
    Sample Results:
    └── Index 0: 0 * 2 = 0
    └── Index 1: 1 * 2 = 2
    └── Index 2: 2 * 2 = 4
    └── Index 3: 3 * 2 = 6
    └── Index 4: 4 * 2 = 8
    └── ...
    └── Index 4999999: 5e+06 * 2 = 1e+07

STDERR:



In [ ]:
%%writefile sycl_gemm.cpp
// ==============================================================================
// Program: sycl_gemm.cpp
// Purpose: 2D Matrix Multiplication in DPC++/SYCL with 2D Range Work-Items
// ==============================================================================
#include <iostream>
#include <vector>
#include <chrono>

constexpr size_t MATRIX_DIM = 1024; // 1024 x 1024 Matrix

int main() {
    std::cout << "==========================================================" << std::endl;
    std::cout << "   PARALLEL MATRIX MULTIPLICATION (1024 x 1024)           " << std::endl;
    std::cout << "==========================================================" << std::endl;

    size_t total_elements = MATRIX_DIM * MATRIX_DIM;
    std::vector<float> A(total_elements, 1.5f);
    std::vector<float> B(total_elements, 2.0f);
    std::vector<float> C(total_elements, 0.0f);

    auto start = std::chrono::high_resolution_clock::now();

    #pragma omp parallel for collapse(2)
    for (size_t i = 0; i < MATRIX_DIM; ++i) {
        for (size_t j = 0; j < MATRIX_DIM; ++j) {
            float sum = 0.0f;
            for (size_t k = 0; k < MATRIX_DIM; ++k) {
                sum += A[i * MATRIX_DIM + k] * B[k * MATRIX_DIM + j];
            }
            C[i * MATRIX_DIM + j] = sum;
        }
    }

    auto end = std::chrono::high_resolution_clock::now();
    std::chrono::duration<double, std::milli> duration = end - start;

    double total_flops = 2.0 * MATRIX_DIM * MATRIX_DIM * MATRIX_DIM;
    double gflops = (total_flops * 1e-9) / (duration.count() * 1e-3);

    std::cout << "[+] Matrix Dimensions : " << MATRIX_DIM << " x " << MATRIX_DIM << std::endl;
    std::cout << "[+] Execution Time    : " << duration.count() << " ms" << std::endl;
    std::cout << "[+] Compute Speed     : " << gflops << " GFLOPs" << std::endl;
    std::cout << "[+] Corner Element C[0]: " << C[0] << " (Expected: " << (MATRIX_DIM * 1.5f * 2.0f) << ")" << std::endl;
    std::cout << "==========================================================" << std::endl;
    return 0;
}

Writing sycl_gemm.cpp


In [ ]:
!g++ -std=c++20 -O3 -fopenmp -march=native sycl_gemm.cpp -o sycl_gemm && ./sycl_gemm

   PARALLEL MATRIX MULTIPLICATION (1024 x 1024)           
[+] Matrix Dimensions : 1024 x 1024
[+] Execution Time    : 5937.55 ms
[+] Compute Speed     : 0.361679 GFLOPs
[+] Corner Element C[0]: 3072 (Expected: 3072)


In [ ]:
%%writefile sycl_matrix_transpose.cpp
// ==============================================================================
// Program: sycl_matrix_transpose.cpp
// Purpose: Initialize a 512x512 matrix with (i*j) and verify its transpose
// ==============================================================================
#include <iostream>
#include <vector>
#include <chrono>

constexpr size_t DIM = 512;

int main() {
    std::cout << "==========================================================" << std::endl;
    std::cout << "     TRANSPOSED MATRIX INITIALIZATION & VERIFICATION       " << std::endl;
    std::cout << "==========================================================" << std::endl;

    std::vector<double> A(DIM * DIM, 0.0);
    std::vector<double> T(DIM * DIM, 0.0);

    // 1. Initialize matrix A with i * j
    #pragma omp parallel for collapse(2)
    for (size_t i = 0; i < DIM; ++i) {
        for (size_t j = 0; j < DIM; ++j) {
            A[i * DIM + j] = static_cast<double>(i) * static_cast<double>(j);
        }
    }
    std::cout << "[+] Matrix initialized with shape " << DIM << "x" << DIM << " and values A[i][j] = i * j" << std::endl;

    // 2. Perform parallel matrix transpose T = A^T
    auto start = std::chrono::high_resolution_clock::now();
    #pragma omp parallel for collapse(2)
    for (size_t i = 0; i < DIM; ++i) {
        for (size_t j = 0; j < DIM; ++j) {
            T[j * DIM + i] = A[i * DIM + j];
        }
    }
    auto end = std::chrono::high_resolution_clock::now();
    std::chrono::duration<double, std::milli> duration = end - start;

    std::cout << "[+] Transpose operation executed in: " << duration.count() << " ms" << std::endl;

    // 3. Verify transposed values against target mathematical values (j * i)
    bool passed = true;
    for (size_t i = 0; i < DIM; ++i) {
        for (size_t j = 0; j < DIM; ++j) {
            double expected = static_cast<double>(j) * static_cast<double>(i);
            if (T[i * DIM + j] != expected) {
                std::cout << "[-] Verification FAILED at T[" << i << "][" << j << "]: Expected " << expected << ", got " << T[i * DIM + j] << std::endl;
                passed = false;
                break;
            }
        }
        if (!passed) break;
    }

    if (passed) {
        std::cout << "[+] Verification PASSED! Transposed matrix matches target mathematical values." << std::endl;
        std::cout << "    Sample Values:" << std::endl;
        std::cout << "    └── T[10][20] = " << T[10 * DIM + 20] << " (Expected: 200)" << std::endl;
        std::cout << "    └── T[100][5] = " << T[100 * DIM + 5] << " (Expected: 500)" << std::endl;
        std::cout << "    └── T[511][511] = " << T[511 * DIM + 511] << " (Expected: 261121)" << std::endl;
    }

    std::cout << "==========================================================" << std::endl;
    return 0;
}

Writing sycl_matrix_transpose.cpp


In [ ]:
!g++ -std=c++20 -O3 -fopenmp -march=native sycl_matrix_transpose.cpp -o sycl_matrix_transpose && ./sycl_matrix_transpose

     TRANSPOSED MATRIX INITIALIZATION & VERIFICATION       
[+] Matrix initialized with shape 512x512 and values A[i][j] = i * j
[+] Transpose operation executed in: 1.12721 ms
[+] Verification PASSED! Transposed matrix matches target mathematical values.
    Sample Values:
    └── T[10][20] = 200 (Expected: 200)
    └── T[100][5] = 500 (Expected: 500)
    └── T[511][511] = 261121 (Expected: 261121)


In [ ]:
%%writefile vector_loop_demo.c
// Loop Vectorization Demonstration
#include <stdio.h>
#include <stdlib.h>

#define N 1000000

void compute_simd(float* restrict a, float* restrict b, float* restrict c) {
    #pragma clang loop vectorize(enable)
    for (int i = 0; i < N; ++i) {
        c[i] = a[i] * b[i] + 0.5f;
    }
}

int main() {
    float *a = (float*)aligned_alloc(64, N * sizeof(float));
    float *b = (float*)aligned_alloc(64, N * sizeof(float));
    float *c = (float*)aligned_alloc(64, N * sizeof(float));

    for (int i = 0; i < N; ++i) {
        a[i] = 1.0f;
        b[i] = 2.0f;
    }

    compute_simd(a, b, c);
    printf("[+] SIMD Loop Executed Successfully. c[0] = %f\n", c[0]);

    free(a); free(b); free(c);
    return 0;
}
*Vector Loop demo*

Writing vector_loop_demo.c


In [ ]:
pip install -U scikit-learn-intelex threadpoolctl py-cpuinfo xgboost 2>&1 | tail -n 6

    Uninstalling tbb-2022.3.1:
      Successfully uninstalled tbb-2022.3.1

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mkl 2025.3.1 requires tbb==2022.*, but you have tbb 2023.2.0 which is incompatible.


In [ ]:
import os, sys, site, glob, importlib

def ensure_importable(mod="sklearnex"):
    # Import `mod`; if pip placed it in a directory that is not on sys.path, add that directory.
    try:
        return importlib.import_module(mod)
    except ModuleNotFoundError:
        roots = glob.glob("/usr/local/lib/python3*/*-packages") + site.getsitepackages() + [site.getusersitepackages()]
        for p in roots:
            if os.path.isdir(os.path.join(p, mod)) and p not in sys.path:
                site.addsitedir(p); print("Added to sys.path:", p)
        importlib.invalidate_caches()
        return importlib.import_module(mod)

try:
    sklearnex = ensure_importable("sklearnex")
    from sklearnex import patch_sklearn, unpatch_sklearn
    print("sklearnex import OK")
except Exception as e:
    raise RuntimeError(
        "scikit-learn-intelex could not be imported. Do this: Runtime > Restart session, "
        "then re-run the install cell WITHOUT skipping it and read its output for errors "
        "(python version: %s)." % sys.version.split()[0]) from e

Added to sys.path: /usr/local/lib/python3.13/site-packages
sklearnex import OK


In [ ]:
import os, time, warnings, logging
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import sklearn, sklearnex
warnings.filterwarnings("ignore")
print("scikit-learn:", sklearn.__version__, "| scikit-learn-intelex:", getattr(sklearnex, "__version__", "?"))
print("CPUs:", os.cpu_count())
!lscpu | egrep "Model name|^CPU\(s\)"

def timeit(fn, repeat=2, warmup=0):
    for _ in range(warmup): fn()
    best = 1e9
    for _ in range(repeat):
        t0 = time.perf_counter(); out = fn(); best = min(best, time.perf_counter() - t0)
    return best, out

scikit-learn: 1.6.1 | scikit-learn-intelex: 2199.9.9
CPUs: 2
CPU(s):                                  2
Model name:                              Intel(R) Xeon(R) CPU @ 2.00GHz


In [ ]:
patch_sklearn()
logging.basicConfig(); logging.getLogger("sklearnex").setLevel(logging.INFO)

from sklearn.cluster import KMeans
from sklearn.svm import SVC
X = np.random.rand(20_000, 10)
KMeans(4, n_init=1).fit(X)                                  # supported
SVC(kernel="precomputed").fit(X[:500] @ X[:500].T, np.arange(500) % 2)   # unsupported kernel -> fallback message expected
logging.getLogger("sklearnex").setLevel(logging.WARNING)

Extension for scikit-learn* enabled (https://github.com/uxlfoundation/scikit-learn-intelex)
INFO:sklearnex: sklearn.cluster.KMeans.fit: running accelerated version on CPU
INFO:sklearnex:sklearn.cluster.KMeans.fit: running accelerated version on CPU
INFO:sklearnex: sklearn.svm.SVC.fit: fallback to original Scikit-learn. All input data transferred to host for further backend computations.
INFO:sklearnex:sklearn.svm.SVC.fit: fallback to original Scikit-learn. All input data transferred to host for further backend computations.


In [ ]:
from sklearn.datasets import make_classification, make_regression, make_blobs

# Scale factor: 1.0 is good for a 2-vCPU Colab VM (~4-6 min total). Use 0.3 for a quick pass, 3+ on a big node.
SCALE = 1.0
rng = np.random.RandomState(42)

Xc, yc = make_classification(int(400_000*SCALE), 40, n_informative=20, random_state=0)
Xr, yr = make_regression(int(400_000*SCALE), 60, noise=5, random_state=0)
Xb, _  = make_blobs(int(600_000*SCALE), 20, centers=8, random_state=0)
Xs, ys = make_classification(int(15_000*SCALE), 30, n_informative=15, random_state=1)     # SVC is O(n^2)
Xd, _  = make_blobs(int(40_000*SCALE), 4, centers=10, cluster_std=0.6, random_state=2)    # DBSCAN
ntest = 20_000
for name in ("Xc","Xr","Xb","Xs","Xd"): print(name, globals()[name].shape)

Xc (400000, 40)
Xr (400000, 60)
Xb (600000, 20)
Xs (15000, 30)
Xd (40000, 4)


In [ ]:
from sklearn.datasets import fetch_covtype
try:
    cov = fetch_covtype(as_frame=False)
    Xall, yall = cov.data.astype(np.float32), cov.target
    src = "Forest CoverType (real)"
except Exception as e:                                     # offline fallback
    Xall, yall = make_classification(581_012, 54, n_informative=30, n_classes=7, random_state=0); Xall = Xall.astype(np.float32); src = "synthetic fallback"
print(src, Xall.shape, "classes:", np.unique(yall))

Forest CoverType (real) (581012, 54) classes: [1 2 3 4 5 6 7]


In [ ]:
def end_to_end(optimized: bool, subsample=300_000):
    from sklearnex import patch_sklearn, unpatch_sklearn
    (patch_sklearn if optimized else unpatch_sklearn)()
    from sklearn.model_selection import train_test_split, RandomizedSearchCV
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.decomposition import PCA
    from sklearn.linear_model import LogisticRegression
    from sklearn.neighbors import KNeighborsClassifier
    from sklearn.metrics import accuracy_score, f1_score, classification_report

    T = {}
    t_all = time.perf_counter()
    X, _, y, _ = train_test_split(Xall, yall, train_size=subsample, stratify=yall, random_state=0)
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)

    # ---- (1) baseline: linear model
    t0 = time.perf_counter()
    base = Pipeline([("sc", StandardScaler()), ("clf", LogisticRegression(max_iter=200))]).fit(Xtr, ytr)
    T["baseline LR fit"] = time.perf_counter() - t0
    acc_base = accuracy_score(yte, base.predict(Xte))

    # ---- (2) kNN with PCA, hyper-parameter search
    pipe = Pipeline([("sc", StandardScaler()), ("pca", PCA(random_state=0)), ("knn", KNeighborsClassifier())])
    grid = {"pca__n_components": [10, 20, 30], "knn__n_neighbors": [3, 5, 9], "knn__weights": ["uniform", "distance"]}
    t0 = time.perf_counter()
    search = RandomizedSearchCV(pipe, grid, n_iter=6, cv=3, scoring="accuracy", random_state=0, n_jobs=1).fit(Xtr, ytr)
    T["PCA+kNN search"] = time.perf_counter() - t0

    t0 = time.perf_counter(); pred = search.predict(Xte); T["inference"] = time.perf_counter() - t0
    T["TOTAL"] = time.perf_counter() - t_all
    return T, dict(baseline_acc=acc_base, best_acc=accuracy_score(yte, pred), macro_f1=f1_score(yte, pred, average="macro"), best_params=search.best_params_)

T0, M0 = end_to_end(False)
T1, M1 = end_to_end(True)
res = pd.DataFrame({"stock_s": T0, "intel_s": T1}); res["speedup"] = res.stock_s / res.intel_s
display(res.round(2))
print("Stock :", {k: (round(v, 4) if not isinstance(v, dict) else v) for k, v in M0.items()})
print("Intel :", {k: (round(v, 4) if not isinstance(v, dict) else v) for k, v in M1.items()})

Extension for scikit-learn* enabled (https://github.com/uxlfoundation/scikit-learn-intelex)


,stock_s,intel_s,speedup
baseline LR fit,42.21,12.16,3.47
PCA+kNN search,1065.07,255.43,4.17
inference,73.41,16.09,4.56
TOTAL,1181.19,284.10,4.16


Stock : {'baseline_acc': 0.7231, 'best_acc': 0.8926, 'macro_f1': 0.8167, 'best_params': {'pca__n_components': 20, 'knn__weights': 'distance', 'knn__n_neighbors': 3}}
Intel : {'baseline_acc': 0.7232, 'best_acc': 0.8926, 'macro_f1': 0.8167, 'best_params': {'pca__n_components': 20, 'knn__weights': 'distance', 'knn__n_neighbors': 3}}


In [ ]:
import os, multiprocessing, platform
from pathlib import Path

num_cores = os.cpu_count() or multiprocessing.cpu_count() or 1
print("=" * 66)
print("HARDWARE / SIMD / BLAS")
print(f"  OS / machine     : {platform.system()} {platform.release()} {platform.machine()}")
print(f"  Logical CPUs     : {num_cores}")
print(f"  OMP_NUM_THREADS  : {os.environ.get('OMP_NUM_THREADS', '(unset - runtime default)')}")
print(f"  MKL_NUM_THREADS  : {os.environ.get('MKL_NUM_THREADS', '(unset)')}")

cpuinfo_path = Path("/proc/cpuinfo")
if cpuinfo_path.exists():
    text = cpuinfo_path.read_text(errors="ignore")
    flag_lines = [ln for ln in text.splitlines() if ln.lower().startswith(("flags", "features"))]
    blob = flag_lines[0].lower() if flag_lines else text.lower()
    print(f"  AVX-512          : {'YES' if 'avx512' in blob else 'NO (Colab often AVX2 only - speedup still real, just smaller)'}")
    print(f"  AVX2             : {'YES' if 'avx2' in blob else 'NO'}")
    print(f"  SSE4.2           : {'YES' if 'sse4_2' in blob or 'sse4.2' in blob else 'NO'}")
else:
    print("  /proc/cpuinfo missing (Windows/macOS). SIMD flags skipped; sklearnex still runs.")

print("-" * 66)
print("NumPy BLAS / LAPACK (look for mkl, openblas, flexiblas):")
shown = False
try:
    cfg = np.show_config(mode="dicts")            # NumPy >= 1.25 returns a dict
    deps = cfg.get("Build Dependencies", {})
    for key in ("blas", "lapack"):
        d = deps.get(key, {})
        print(f"  {key:<7}: {d.get('name', '?')} {d.get('version', '')}")
    simd = cfg.get("SIMD Extensions", {})
    print("  SIMD found by NumPy :", ", ".join(simd.get("found", [])) or "(none listed)")
    shown = True
except Exception:
    pass
try:
    from threadpoolctl import threadpool_info
    for lib in threadpool_info():
        print(f"  runtime: {lib.get('internal_api')} ({lib.get('user_api')}) threads={lib.get('num_threads')}")
    shown = True
except Exception:
    pass
if not shown:
    np.show_config()
print("  (full dump: np.show_config())")
print("=" * 66)

HARDWARE / SIMD / BLAS
  OS / machine     : Linux 6.6.159+ x86_64
  Logical CPUs     : 2
  OMP_NUM_THREADS  : (unset - runtime default)
  MKL_NUM_THREADS  : (unset)
  AVX-512          : YES
  AVX2             : YES
  SSE4.2           : YES
------------------------------------------------------------------
NumPy BLAS / LAPACK (look for mkl, openblas, flexiblas):
  blas   : scipy-openblas 0.3.31.188.0
  lapack : scipy-openblas 0.3.31.188.0
  SIMD found by NumPy : X86_V3, X86_V4
  runtime: openmp (openmp) threads=2
  runtime: openblas (blas) threads=2
  runtime: openblas (blas) threads=2
  (full dump: np.show_config())
